In [1]:
import os
from pathlib import Path

if Path.cwd().name == "notebooks":
    os.chdir("..")

In [2]:
import json

import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer

from mini_rag.bm25 import BM25
from mini_rag.dense import DenseSearch

c:\Users\vmaneata\Work\mini-rag\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
with open("eval/eval_draft.jsonl", "r", encoding="utf-8") as f:
    eval_questions = [json.loads(line) for line in f]
with open("data/chunks.jsonl", "r", encoding="utf-8") as f:
    chunks = [json.loads(line) for line in f]

In [4]:
bm25 = BM25(chunks, stem=False)
model = SentenceTransformer("all-MiniLM-L6-v2")
dense = DenseSearch(chunks, model)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5561.88it/s]


In [5]:
def recall_at_k(df, k):
    found_at_k = df[(df["Prediction"] == "found") & (df["Rank"] <= k)][
        "Prediction"
    ].count()
    total = df["Prediction"].count()
    return found_at_k / total if total > 0 else 0

In [6]:
results_for_bm25 = []

for question in eval_questions:
    query = question["question"]
    source_id = question["source_id"]
    bm25_results = bm25.search(query, k=10)
    for index, (chunk, _) in enumerate(bm25_results):
        if source_id == chunk["id"]:
            results_for_bm25.append(
                {"question": query, "Prediction": "found", "Rank": index + 1}
            )
            break
    else:
        results_for_bm25.append({"question": query, "Prediction": "missed", "Rank": None})

In [7]:
results_for_dense = []

for question in eval_questions:
    query = question["question"]
    source_id = question["source_id"]
    dense_results = dense.search(query, top_k=10)
    for index, (chunk, _) in enumerate(dense_results):
        if source_id == chunk["id"]:
            results_for_dense.append(
                {"question": query, "Prediction": "found", "Rank": index + 1}
            )
            break
    else:
        results_for_dense.append({"question": query, "Prediction": "missed", "Rank": None})


In [8]:
df_bm25 = pd.DataFrame(results_for_bm25)
df_dense = pd.DataFrame(results_for_dense)

In [9]:

df_dense.head()

,question,Prediction,Rank
0,Where can I find the auto-generated interactiv...,found,2.0
1,How can I stop FastAPI from sending exception ...,found,1.0
2,How can I securely store user passwords in a F...,found,3.0
3,"In programming terms, what are the different m...",found,1.0
4,Why do programs call operations like network r...,found,8.0


In [10]:
df_bm25.head()

,question,Prediction,Rank
0,Where can I find the auto-generated interactiv...,found,2.0
1,How can I stop FastAPI from sending exception ...,found,1.0
2,How can I securely store user passwords in a F...,found,1.0
3,"In programming terms, what are the different m...",found,1.0
4,Why do programs call operations like network r...,found,1.0


In [11]:
both = pd.merge(df_bm25, df_dense, on="question", suffixes=("_bm25", "_dense"))
both.head()

,question,Prediction_bm25,Rank_bm25,Prediction_dense,Rank_dense
0,Where can I find the auto-generated interactiv...,found,2.0,found,2.0
1,How can I stop FastAPI from sending exception ...,found,1.0,found,1.0
2,How can I securely store user passwords in a F...,found,1.0,found,3.0
3,"In programming terms, what are the different m...",found,1.0,found,1.0
4,Why do programs call operations like network r...,found,1.0,found,8.0


In [12]:
bm25_hit = both["Rank_bm25"] <= 10
dense_hit = both["Rank_dense"] <= 10

In [13]:
pd.crosstab(bm25_hit, dense_hit)

Rank_dense,False,True
Rank_bm25,,
False,6,4
True,3,29


In [14]:
from mini_rag.hybrid import HybridSearch

In [15]:
hybrid_search = HybridSearch(bm25, dense)

In [16]:
results_for_hybrid = []

for question in eval_questions:
    query = question["question"]
    source_id = question["source_id"]
    hybrid_results = hybrid_search.search(query, 10)
    for index, (chunk, _) in enumerate(hybrid_results):
        if source_id == chunk["id"]:
            results_for_hybrid.append(
                {"question": query, "Prediction": "found", "Rank": index + 1}
            )
            break
    else:
        results_for_hybrid.append({"question": query, "Prediction": "missed", "Rank": None})

In [17]:
df_hybrid = pd.DataFrame(results_for_hybrid)
df_hybrid.head()

,question,Prediction,Rank
0,Where can I find the auto-generated interactiv...,found,1.0
1,How can I stop FastAPI from sending exception ...,found,1.0
2,How can I securely store user passwords in a F...,found,1.0
3,"In programming terms, what are the different m...",found,1.0
4,Why do programs call operations like network r...,found,2.0


In [18]:
for k in range(1, 11):
    print(f"Hybrid Recall at {k}: {recall_at_k(df_hybrid, k)}")

Hybrid Recall at 1: 0.5952380952380952
Hybrid Recall at 2: 0.7142857142857143
Hybrid Recall at 3: 0.7380952380952381
Hybrid Recall at 4: 0.7619047619047619
Hybrid Recall at 5: 0.7857142857142857
Hybrid Recall at 6: 0.7857142857142857
Hybrid Recall at 7: 0.7857142857142857
Hybrid Recall at 8: 0.7857142857142857
Hybrid Recall at 9: 0.8571428571428571
Hybrid Recall at 10: 0.8571428571428571


In [19]:
mrr_hybrid = (1 / df_hybrid["Rank"]).fillna(0).mean()
print(f"Hybrid Mean Reciprocal Rank: {mrr_hybrid}")

Hybrid Mean Reciprocal Rank: 0.6813492063492064


In [20]:
ndcg10_hybrid = pd.Series(1 / np.log2(df_hybrid["Rank"] + 1)).fillna(0).mean()
print(f"Hybrid Normalized Discounted Cumulative Gain: {ndcg10_hybrid}")

Hybrid Normalized Discounted Cumulative Gain: 0.7232206695275516


In [26]:
diff_bm25 = []
diff_hybrid = []

for question in eval_questions:
    query = question['question']
    source_id = question['source_id']
    bm25_result = bm25.search(query, k=10)
    hybrid_results = hybrid_search.search(query, k=10)
    for index, (chunk, _) in enumerate(bm25_result):
        if source_id == chunk["id"]:
            diff_bm25.append(
                {"question": query, "Prediction": "found", "Rank": index + 1}
            )
            break
    else:
        diff_bm25.append(
            {"question": query, "Prediction": "not found", "Rank": None}
        )
    for index, (chunk, _) in enumerate(hybrid_results):
        if source_id == chunk["id"]:
            diff_hybrid.append(
                {"question": query, "Prediction": "found (hybrid)", "Rank": index + 1}
            )
            break
    else:
        diff_hybrid.append(
            {"question": query, "Prediction": "not found (hybrid)", "Rank": None}
        )
diff_bm25 = pd.DataFrame(diff_bm25)
diff_hybrid = pd.DataFrame(diff_hybrid )

In [27]:
diff_bm25["RR"] = 1 / diff_bm25["Rank"]
diff_bm25["RR"] = diff_bm25["RR"].fillna(0)
diff_bm25.head()
diff_hybrid["RR"] = 1 / diff_hybrid["Rank"]
diff_hybrid["RR"] = diff_hybrid["RR"].fillna(0)
diff_hybrid.head()

,question,Prediction,Rank,RR
0,Where can I find the auto-generated interactiv...,found (hybrid),1.0,1.0
1,How can I stop FastAPI from sending exception ...,found (hybrid),1.0,1.0
2,How can I securely store user passwords in a F...,found (hybrid),1.0,1.0
3,"In programming terms, what are the different m...",found (hybrid),1.0,1.0
4,Why do programs call operations like network r...,found (hybrid),2.0,0.5


In [28]:
diff = pd.merge(diff_bm25, diff_hybrid, on="question", how="outer", suffixes=("_bm25", "_hybrid"))
diff.head()

,question,Prediction_bm25,Rank_bm25,RR_bm25,Prediction_hybrid,Rank_hybrid,RR_hybrid
0,Are translations for the FastAPI docs still wr...,found,1.0,1.000000,found (hybrid),1.0,1.000000
1,Before the async/await syntax existed in Pytho...,found,7.0,0.142857,found (hybrid),3.0,0.333333
2,Can I use a GraphQL library together with regu...,found,1.0,1.000000,found (hybrid),1.0,1.000000
3,Does FastAPI's main application class build on...,found,3.0,0.333333,found (hybrid),9.0,0.111111
4,How can I confirm that FastAPI generates separ...,not found,NaN,0.000000,not found (hybrid),NaN,0.000000


In [29]:
d = (diff["RR_hybrid"] - diff["RR_bm25"]).to_numpy()

In [31]:
observed = d.mean()

In [36]:
rng = np.random.default_rng(42)
signs = rng.choice([-1, 1], size=len(d))

In [37]:
signs

array([-1,  1,  1, -1, -1,  1, -1,  1, -1, -1,  1,  1,  1,  1,  1,  1,  1,
       -1,  1, -1,  1, -1, -1,  1,  1,  1, -1,  1,  1, -1, -1, -1, -1,  1,
        1, -1,  1,  1, -1,  1, -1,  1])

In [35]:
(signs * d).mean()

np.float64(0.06643990929705215)

In [38]:
sims =[]
for _ in range(10000): 
    signs = rng.choice([-1, 1], size=len(d))
    permuted_mean = (signs * d).mean()
    sims.append(permuted_mean)

sims = np.array(sims)
p = np.mean(np.abs(sims) >= abs(observed))
p

np.float64(0.1343)

In [42]:
s=d.std(ddof=1)
n=len(d)
se = s / np.sqrt(n)
t_stat = observed / se
from scipy import stats
p_t = 2 * stats.t.sf(abs(t_stat), df=n - 1)

In [43]:
p_t

np.float64(0.13145579876990224)